# Reintegration Timeline: Causal Pipeline
### IS 455 — Machine Learning | CRISP-DM Framework

**Business Problem:** Which resident characteristics and early-stay indicators are associated with longer or shorter time to case closure? Can we identify which factors most influence how long a girl stays before reintegration?

**Who cares:** Case managers and social workers who need to understand which intake and early-stay factors predict reintegration timelines, so they can allocate resources and set realistic expectations for residents and families.

**Approach:** Explanatory (causal) OLS regression. The goal is to quantify relationships between intake characteristics and early-stay indicators with `days_stayed`, not to generate out-of-sample predictions.

> **Predictive vs. Explanatory:** This pipeline is explicitly **explanatory**. We want to understand *which factors are associated with longer or shorter stays and by how much* — not to predict exact days for new residents. With n=30, any predictive pipeline would be severely overfit and unreliable. The causal framing is honest and still operationally valuable.

> **Sample size caveat:** Only 30 of 60 residents have a `date_closed`, meaning the training set is n=30. This is documented throughout and should be revisited as more cases are closed.

---
## Phase 1 — Business Understanding

### Target Variable
`days_stayed` = `date_closed` − `date_of_admission` for residents with a closed case.

This represents total case duration. It is a proxy for reintegration timeline — not all closed cases end in successful reintegration (some are On Hold or transferred), but closure is the operationally available endpoint.

### Early-Stay Window: 90 Days
Features from longitudinal tables (health, education, counseling, visitations) are restricted to records within the **first 90 days** of admission. This prevents reverse causation — we do not want features that reflect how long someone stayed to predict how long they stayed.

- 90-day window chosen because: minimum `days_stayed` is 191 days (so all residents have 90-day records), and all longitudinal tables achieve 30/30 coverage at 90 days
- Incidents excluded — only 7/30 residents had any incident in the first 90 days (too sparse)

### Feature Sources
| Source | Features | Rationale |
|---|---|---|
| `residents.csv` | Intake demographics, risk level, case category, sub-categories, family profile, reintegration type | Available at admission, no reverse causation risk |
| `health_wellbeing_records.csv` | Early avg health, nutrition, sleep, energy, psych checkups | Physical recovery trajectory in first 90 days |
| `education_records.csv` | Early attendance rate, education progress | Educational stability early in stay |
| `process_recordings.csv` | Session count, % progress noted, % concerns flagged, % referrals made | Counseling trajectory — key social welfare indicator |
| `home_visitations.csv` | Visit count, % safety concerns, % favorable outcomes, % cooperative family | Family readiness for reintegration |
| `intervention_plans.csv` | Has reintegration plan, has legal plan | Structural planning indicators |

### Excluded
- `incident_reports` — only 7/30 residents had any early-stay incident (too sparse to model)
- `current_risk_level` — changes over time, not an intake feature
- Any feature computed over the full stay (reverse causation)

---
## Phase 2 — Data Understanding

In [1]:
import warnings
import json
import os
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import SelectKBest, f_regression
import joblib

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Data source config ───────────────────────────────────────────────────────
USE_DB = True

# CSV fallback
DEFAULT_DATA_DIR = Path.home() / "Downloads" / "lighthouse_csv_v7"
DATA_DIR = Path(os.environ.get("LIGHTHOUSE_CSV_DIR", str(DEFAULT_DATA_DIR))).expanduser().resolve()

OUTPUT_DIR = Path("model_outputs_reintegration")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EARLY_WINDOW_DAYS = 90


def _to_snake(name: str) -> str:
    import re

    name = name.replace(" ", "_").replace("-", "_")
    s1 = re.sub("(.)([A-Z][a-z]+)", r"\1_\2", name)
    s2 = re.sub("([a-z0-9])([A-Z])", r"\1_\2", s1)
    s2 = re.sub(r"__+", "_", s2)
    return s2.strip("_").lower()


def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [_to_snake(c) for c in df.columns]
    return df


def _find_dotenv(start: Optional[Path] = None) -> Optional[Path]:
    here = (start or Path.cwd()).resolve()
    for p in [here, *here.parents]:
        candidate = p / ".env"
        if candidate.exists():
            return candidate
    return None


def _make_sqlalchemy_url(db_override: Optional[str] = None) -> str:
    host = os.getenv('PGHOST')
    user = os.getenv('PGUSER')
    port = os.getenv('PGPORT', '5432')
    db   = db_override or os.getenv('PGDATABASE')
    pwd  = os.getenv('PGPASSWORD')
    if not all([host, user, db, pwd]):
        missing = [k for k in ['PGHOST','PGUSER','PGDATABASE','PGPASSWORD'] if not os.getenv(k)]
        raise ValueError(f'Missing required env vars (expected in .env): {missing}')
    return f'postgresql+psycopg2://{user}:{pwd}@{host}:{port}/{db}?sslmode=require'


def _pip_install(packages):
    import sys
    import subprocess

    cmd = [sys.executable, "-m", "pip", "install", "--user", *packages]
    print("Installing missing packages:", " ".join(packages))
    subprocess.check_call(cmd)


def _make_engine():
    try:
        from sqlalchemy import create_engine
    except ImportError:
        _pip_install(["sqlalchemy", "psycopg2-binary"])
        from sqlalchemy import create_engine

    try:
        from dotenv import load_dotenv
    except ImportError:
        _pip_install(["python-dotenv"])
        from dotenv import load_dotenv

    env_path = _find_dotenv()
    if env_path is None:
        raise FileNotFoundError("No .env found (expected in project folder or parents).")

    load_dotenv(env_path, override=False)
    url = _make_sqlalchemy_url(db_override=os.getenv("PGDATABASE_OVERRIDE"))
    return create_engine(url, pool_pre_ping=True, connect_args={"options": "-c statement_timeout=20000"})


def _read_table(engine, logical_name: str) -> pd.DataFrame:
    schema = os.getenv("PGSCHEMA", "public")
    TABLES = {
        "residents": "Residents",
        "health_wellbeing_records": "HealthWellbeingRecords",
        "education_records": "EducationRecords",
        "process_recordings": "ProcessRecordings",
        "home_visitations": "HomeVisitations",
        "intervention_plans": "InterventionPlans",
    }
    table = TABLES[logical_name]
    df = pd.read_sql_query(f'SELECT * FROM "{schema}"."{table}"', con=engine)
    return _normalize_columns(df)


def _to_utc_naive(s: pd.Series) -> pd.Series:
    """Convert mixed tz-aware/naive datetimes to tz-naive UTC."""
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    return dt.dt.tz_convert(None)


print("Libraries loaded ✓")
print(f"Early-stay window: {EARLY_WINDOW_DAYS} days")
print(f"USE_DB={USE_DB}  |  DATA_DIR={DATA_DIR}")

Libraries loaded ✓
Early-stay window: 90 days
USE_DB=True  |  DATA_DIR=/Users/jamescorrigan/Downloads/lighthouse_csv_v7


In [2]:
# ── Load residents and compute target ───────────────────────────────────────
if USE_DB:
    engine = _make_engine()
    residents = _read_table(engine, "residents")
else:
    if not DATA_DIR.exists():
        raise FileNotFoundError(
            f"DATA_DIR not found: {DATA_DIR}. Set LIGHTHOUSE_CSV_DIR or place CSVs in {DEFAULT_DATA_DIR}."
        )
    residents = pd.read_csv(DATA_DIR / "residents.csv")

residents["date_of_admission"] = _to_utc_naive(residents["date_of_admission"])
residents["date_closed"] = _to_utc_naive(residents["date_closed"])

print(f"Total residents: {len(residents)}")
print(f"  With date_closed (usable): {residents['date_closed'].notna().sum()}")
print(f"  Without date_closed (active/no endpoint): {residents['date_closed'].isna().sum()}")
print()
print("case_status breakdown:")
print(residents["case_status"].value_counts().to_string())
print()
print("reintegration_status breakdown (closed cases only):")
closed_mask = residents["date_closed"].notna()
print(residents.loc[closed_mask, "reintegration_status"].value_counts().to_string())

Total residents: 60
  With date_closed (usable): 30
  Without date_closed (active/no endpoint): 30

case_status breakdown:
case_status
Active         30
Closed         19
Transferred    11

reintegration_status breakdown (closed cases only):
reintegration_status
Completed      12
In Progress    12
On Hold         3
Not Started     3


In [3]:
# ── Filter to closed cases and compute days_stayed ──────────────────────────
closed = residents[residents['date_closed'].notna()].copy()
closed['days_stayed'] = (closed['date_closed'] - closed['date_of_admission']).dt.days
closed_ids  = closed['resident_id'].tolist()
admit_map   = closed.set_index('resident_id')['date_of_admission'].to_dict()

print(f'Modelling dataset: n={len(closed)} closed residents')
print()
print('days_stayed distribution:')
print(closed['days_stayed'].describe().round(1).to_string())

Modelling dataset: n=30 closed residents

days_stayed distribution:
count     30.0
mean     445.1
std      162.4
min      191.0
25%      286.8
50%      431.0
75%      583.0
max      697.0


In [4]:
# ── Target distribution plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(closed['days_stayed'], bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(closed['days_stayed'].mean(), color='red', linestyle='--', label=f'Mean={closed["days_stayed"].mean():.0f}d')
axes[0].axvline(closed['days_stayed'].median(), color='orange', linestyle='--', label=f'Median={closed["days_stayed"].median():.0f}d')
axes[0].set_title('Distribution of days_stayed (n=30)')
axes[0].set_xlabel('Days Stayed')
axes[0].set_ylabel('Count')
axes[0].legend()

axes[1].boxplot(closed['days_stayed'], vert=True)
axes[1].set_title('days_stayed — Boxplot')
axes[1].set_ylabel('Days Stayed')
axes[1].set_xticks([])

plt.suptitle('Target Variable: Days from Admission to Case Closure', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/target_distribution.png', dpi=120)
plt.show()
print('Saved: target_distribution.png')

Saved: target_distribution.png


In [5]:
# ── Early-stay window coverage check ────────────────────────────────────────
table_configs = [
    ('health_wellbeing_records.csv',  'record_date'),
    ('education_records.csv',          'record_date'),
    ('process_recordings.csv',         'session_date'),
    ('home_visitations.csv',           'visit_date'),
    ('intervention_plans.csv',         'created_at'),
]

print(f"Coverage at {EARLY_WINDOW_DAYS}-day window:")
for fname, date_col in table_configs:
    if USE_DB:
        logical = fname.replace('.csv', '')
        t = _read_table(engine, logical)
    else:
        t = pd.read_csv(DATA_DIR / fname)

    t = t[t["resident_id"].isin(closed_ids)].copy()
    t[date_col] = _to_utc_naive(t[date_col])
    t["days_since"] = (t[date_col] - t["resident_id"].map(admit_map)).dt.days
    in_window = t[t["days_since"] <= EARLY_WINDOW_DAYS]
    covered = in_window["resident_id"].nunique()
    print(f"  {fname:<40} {len(in_window):>5} records | {covered}/30 residents covered")

Coverage at 90-day window:
  health_wellbeing_records.csv               118 records | 30/30 residents covered
  education_records.csv                      118 records | 30/30 residents covered
  process_recordings.csv                     204 records | 30/30 residents covered
  home_visitations.csv                       117 records | 29/30 residents covered
  intervention_plans.csv                      90 records | 30/30 residents covered


---
## Phase 3 — Data Preparation

In [6]:
def get_early_records(fname, date_col, closed_ids, admit_map, window=EARLY_WINDOW_DAYS):
    """Load a table, filter to closed residents, and restrict to early-stay window."""
    if USE_DB:
        logical = fname.replace('.csv', '')
        t = _read_table(engine, logical)
    else:
        t = pd.read_csv(DATA_DIR / fname)
    t = t[t["resident_id"].isin(closed_ids)].copy()
    t[date_col] = _to_utc_naive(t[date_col])
    t["days_since_admit"] = (t[date_col] - t["resident_id"].map(admit_map)).dt.days
    return t[t["days_since_admit"] <= window]


def parse_age_string(s):
    """Parse 'X Years Y months' → decimal years."""
    try:
        parts = str(s).lower().split()
        years  = int(parts[0])
        months = int(parts[2]) if len(parts) >= 4 else 0
        return round(years + months / 12, 2)
    except:
        return np.nan


print('Helper functions defined ✓')

Helper functions defined ✓


In [7]:
# ── Health features (first 90 days) ─────────────────────────────────────────
h = get_early_records('health_wellbeing_records.csv', 'record_date', closed_ids, admit_map)
h_agg = h.groupby('resident_id').agg(
    early_avg_health     = ('general_health_score',    'mean'),
    early_avg_nutrition  = ('nutrition_score',          'mean'),
    early_avg_sleep      = ('sleep_quality_score',      'mean'),
    early_avg_energy     = ('energy_level_score',       'mean'),
    early_psych_checkups = ('psychological_checkup_done','sum'),
).round(3)
print(f'Health features — {len(h_agg)} residents, {len(h)} records')
print(h_agg.describe().round(3).to_string())

Health features — 30 residents, 118 records
       early_avg_health  early_avg_nutrition  early_avg_sleep  early_avg_energy  early_psych_checkups
count            30.000               30.000           30.000            30.000                30.000
mean              3.079                3.132            3.033             2.899                 1.967
std               0.147                0.138            0.137             0.144                 0.890
min               2.658                2.620            2.548             2.542                 0.000
25%               2.982                3.081            2.966             2.810                 1.250
50%               3.068                3.156            3.031             2.888                 2.000
75%               3.189                3.200            3.118             2.970                 2.000
max               3.330                3.407            3.302             3.218                 4.000


In [8]:
# ── Education features (first 90 days) ──────────────────────────────────────
e = get_early_records('education_records.csv', 'record_date', closed_ids, admit_map)
e_agg = e.groupby('resident_id').agg(
    early_avg_attendance = ('attendance_rate',   'mean'),
    early_avg_progress   = ('progress_percent',  'mean'),
).round(3)
print(f'Education features — {len(e_agg)} residents, {len(e)} records')
print(e_agg.describe().round(3).to_string())

Education features — 30 residents, 118 records
       early_avg_attendance  early_avg_progress
count                30.000              30.000
mean                  0.702              64.887
std                   0.057              15.475
min                   0.574              27.075
25%                   0.667              54.225
50%                   0.706              67.550
75%                   0.741              74.694
max                   0.799              95.250


In [9]:
# ── Process recording features (first 90 days) ───────────────────────────────
r = get_early_records('process_recordings.csv', 'session_date', closed_ids, admit_map)
r_agg = r.groupby('resident_id').agg(
    early_session_count    = ('recording_id',      'count'),
    early_pct_progress     = ('progress_noted',     'mean'),
    early_pct_concerns     = ('concerns_flagged',   'mean'),
    early_pct_referral     = ('referral_made',      'mean'),
).round(3)
print(f'Process recording features — {len(r_agg)} residents, {len(r)} records')
print(r_agg.describe().round(3).to_string())

Process recording features — 30 residents, 204 records
       early_session_count  early_pct_progress  early_pct_concerns  early_pct_referral
count               30.000              30.000              30.000              30.000
mean                 6.800               0.948               0.220               0.172
std                  2.845               0.092               0.177               0.189
min                  2.000               0.667               0.000               0.000
25%                  5.000               0.913               0.085               0.000
50%                  6.000               1.000               0.218               0.167
75%                  8.750               1.000               0.333               0.250
max                 14.000               1.000               0.600               0.750


In [10]:
# ── Home visitation features (first 90 days) ─────────────────────────────────
v = get_early_records('home_visitations.csv', 'visit_date', closed_ids, admit_map)
v_agg = v.groupby('resident_id').agg(
    early_visit_count        = ('visitation_id',           'count'),
    early_pct_safety_concern = ('safety_concerns_noted',   'mean'),
    early_pct_favorable      = ('visit_outcome',           lambda x: (x == 'Favorable').mean()),
    early_pct_cooperative    = ('family_cooperation_level', lambda x: x.isin(['Highly Cooperative','Cooperative']).mean()),
).round(3)
print(f'Home visitation features — {len(v_agg)} residents, {len(v)} records')
print(v_agg.describe().round(3).to_string())

Home visitation features — 29 residents, 117 records
       early_visit_count  early_pct_safety_concern  early_pct_favorable  early_pct_cooperative
count             29.000                    29.000               29.000                  29.00
mean               4.034                     0.239                0.406                   0.45
std                1.842                     0.181                0.315                   0.31
min                1.000                     0.000                0.000                   0.00
25%                2.000                     0.000                0.200                   0.25
50%                4.000                     0.250                0.333                   0.50
75%                5.000                     0.333                0.667                   0.60
max                8.000                     0.500                1.000                   1.00


In [11]:
# ── Intervention plan features (all within 90 days confirmed) ────────────────
p = get_early_records('intervention_plans.csv', 'created_at', closed_ids, admit_map)
p_agg = p.groupby('resident_id').agg(
    early_has_reintegration_plan = ('plan_category', lambda x: int('Reintegration' in x.values)),
    early_has_legal_plan         = ('plan_category', lambda x: int('Legal' in x.values)),
).astype(int)
print(f'Intervention plan features — {len(p_agg)} residents')
print(p_agg.value_counts().to_string())

Intervention plan features — 30 residents
early_has_reintegration_plan  early_has_legal_plan
0                             0                       30


In [12]:
# ── Intake features from residents.csv / Residents table ─────────────────────
# CSV had age_upon_admission as a string; DB has date_of_birth + date_of_admission.
if 'age_upon_admission' in closed.columns:
    closed['age_admit_years'] = closed['age_upon_admission'].apply(parse_age_string)
else:
    dob = pd.to_datetime(closed.get('date_of_birth'), errors='coerce')
    doa = pd.to_datetime(closed.get('date_of_admission'), errors='coerce')
    closed['age_admit_years'] = ((doa - dob).dt.days / 365.25).round(2)

closed['reintegration_type'] = closed.get('reintegration_type', pd.Series(index=closed.index, dtype=object)).fillna('Unknown')

intake_cols = [
    'resident_id', 'days_stayed', 'age_admit_years',
    'initial_risk_level', 'reintegration_type', 'referral_source', 'case_category',
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_orphaned', 'sub_cat_child_labor',
    'is_pwd', 'has_special_needs',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous', 'family_informal_settler'
]

# If DB schema omits some columns (e.g., sub-categories collapsed), keep only what exists.
intake_cols = [c for c in intake_cols if c in closed.columns]
master = closed[intake_cols].copy()

# Merge all aggregated tables
for agg_df in [h_agg, e_agg, r_agg, v_agg, p_agg]:
    master = master.merge(agg_df, on='resident_id', how='left')

# Fill the 1 resident with no early visitations with 0 (zero visits is meaningful data)
visit_cols = ['early_visit_count','early_pct_safety_concern','early_pct_favorable','early_pct_cooperative']
master[visit_cols] = master[visit_cols].fillna(0)

print(f'Master dataset shape: {master.shape}')
print(f'Missing values:')
missing = master.isnull().sum()
print(missing[missing > 0].to_string() if missing.any() else '  None ✓')

Master dataset shape: (30, 36)
Missing values:
  None ✓


In [13]:
# ── One-hot encode categoricals, standardize numerics ───────────────────────
cat_cols  = ['initial_risk_level', 'reintegration_type', 'referral_source', 'case_category']
bool_cols = [
    'sub_cat_trafficked', 'sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
    'sub_cat_osaec', 'sub_cat_at_risk', 'sub_cat_orphaned', 'sub_cat_child_labor',
    'is_pwd', 'has_special_needs',
    'family_is_4ps', 'family_solo_parent', 'family_indigenous', 'family_informal_settler',
    'early_has_reintegration_plan', 'early_has_legal_plan'
]
num_cols  = [
    'age_admit_years',
    'early_avg_health', 'early_avg_nutrition', 'early_avg_sleep',
    'early_avg_energy', 'early_psych_checkups',
    'early_avg_attendance', 'early_avg_progress',
    'early_session_count', 'early_pct_progress', 'early_pct_concerns', 'early_pct_referral',
    'early_visit_count', 'early_pct_safety_concern', 'early_pct_favorable', 'early_pct_cooperative'
]

# DB schema may omit CSV-only columns already dropped from master (e.g. family_is_4ps).
cat_cols = [c for c in cat_cols if c in master.columns]
bool_cols = [c for c in bool_cols if c in master.columns]
num_cols = [c for c in num_cols if c in master.columns]

# One-hot encode — drop most common category as reference
ref_cats      = {}
encoded_parts = []
for col in cat_cols:
    ref = master[col].value_counts().idxmax()
    ref_cats[col] = ref
    dummies = pd.get_dummies(master[col], prefix=col)
    if f'{col}_{ref}' in dummies.columns:
        dummies = dummies.drop(columns=[f'{col}_{ref}'])
    encoded_parts.append(dummies.reset_index(drop=True))
    print(f'  {col}: reference="{ref}", dummies={list(dummies.columns)}')

bool_df = master[bool_cols].astype(int).reset_index(drop=True)

# Standardize numerics (mean=0, std=1) for coefficient comparability
num_df = master[num_cols].copy().reset_index(drop=True)
num_stats = {}
for col in num_cols:
    mu, sigma = num_df[col].mean(), num_df[col].std()
    num_stats[col] = {'mean': mu, 'std': sigma}
    if sigma > 0:
        num_df[col] = (num_df[col] - mu) / sigma

X_full = pd.concat(encoded_parts + [bool_df, num_df], axis=1).astype(float)
y      = master['days_stayed'].reset_index(drop=True).astype(float)

print(f'\nFull feature matrix: {X_full.shape[0]} rows × {X_full.shape[1]} features')
print(f'Target (days_stayed): mean={y.mean():.1f}, std={y.std():.1f}, range=[{y.min()},{y.max()}]')

  initial_risk_level: reference="Medium", dummies=['initial_risk_level_Critical', 'initial_risk_level_High', 'initial_risk_level_Low']
  reintegration_type: reference="Foster Care", dummies=['reintegration_type_Adoption (Domestic)', 'reintegration_type_Adoption (Inter-Country)', 'reintegration_type_Family Reunification', 'reintegration_type_Independent Living', 'reintegration_type_None']
  referral_source: reference="Government Agency", dummies=['referral_source_Community', 'referral_source_Court Order', 'referral_source_NGO', 'referral_source_Police', 'referral_source_Self-Referral']
  case_category: reference="Abandoned", dummies=['case_category_Foundling', 'case_category_Neglected', 'case_category_Surrendered']

Full feature matrix: 30 rows × 46 features
Target (days_stayed): mean=445.1, std=162.4, range=[191.0,697.0]


---
## Phase 4 — Exploration

In [14]:
# ── Numeric correlations with days_stayed ────────────────────────────────────
corr_df = master[num_cols + ['days_stayed']].corr()['days_stayed'].drop('days_stayed')
corr_df = corr_df.sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ['steelblue' if v > 0 else 'tomato' for v in corr_df.values]
ax.barh(corr_df.index[::-1], corr_df.values[::-1], color=colors[::-1])
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with days_stayed')
ax.set_title('Numeric Feature Correlations with days_stayed')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/feature_correlations.png', dpi=120)
plt.show()

print('Correlations with days_stayed (sorted by absolute value):')
print(corr_df.round(3).to_string())

Correlations with days_stayed (sorted by absolute value):
early_pct_concerns         -0.408
early_pct_safety_concern   -0.218
early_avg_progress          0.149
early_pct_favorable         0.140
early_avg_nutrition         0.135
early_avg_sleep             0.108
early_pct_referral          0.108
early_avg_energy            0.090
early_visit_count           0.079
early_psych_checkups       -0.068
early_avg_health           -0.057
early_pct_progress          0.057
early_avg_attendance        0.053
early_pct_cooperative      -0.029
age_admit_years            -0.025
early_session_count        -0.009


In [15]:
# ── Categorical breakdowns ────────────────────────────────────────────────────
for col in cat_cols:
    means = master.groupby(col)['days_stayed'].agg(['mean','count']).round(1)
    means.columns = ['mean_days', 'n']
    means = means.sort_values('mean_days', ascending=False)
    print(f'\n{col} (reference=\'{ref_cats[col]}\'):')
    print(means.to_string())


initial_risk_level (reference='Medium'):
                    mean_days   n
initial_risk_level               
Critical                514.0   2
Low                     460.0   7
High                    434.6   7
Medium                  433.1  14

reintegration_type (reference='Foster Care'):
                          mean_days  n
reintegration_type                    
Independent Living            645.0  3
Adoption (Domestic)           491.7  7
None                          438.0  2
Family Reunification          418.6  7
Foster Care                   416.9  8
Adoption (Inter-Country)      278.3  3

referral_source (reference='Government Agency'):
                   mean_days  n
referral_source                
Police                 526.2  5
Community              513.6  5
Court Order            456.8  4
NGO                    433.3  3
Self-Referral          397.4  5
Government Agency      380.0  8

case_category (reference='Abandoned'):
               mean_days   n
case_category       

In [16]:
# ── Boolean feature means ─────────────────────────────────────────────────────
print('Mean days_stayed by boolean feature (True vs False):')
for col in bool_cols:
    t_mean = master[master[col] == True]['days_stayed'].mean()
    f_mean = master[master[col] == False]['days_stayed'].mean()
    t_n    = master[col].sum()
    diff   = t_mean - f_mean
    print(f'  {col:<35} True={t_mean:.0f}d (n={int(t_n)})  False={f_mean:.0f}d  diff={diff:+.0f}d')

Mean days_stayed by boolean feature (True vs False):
  sub_cat_trafficked                  True=414d (n=5)  False=451d  diff=-37d
  sub_cat_physical_abuse              True=435d (n=6)  False=448d  diff=-13d
  sub_cat_sexual_abuse                True=383d (n=8)  False=468d  diff=-84d
  sub_cat_osaec                       True=504d (n=4)  False=436d  diff=+67d
  sub_cat_at_risk                     True=530d (n=6)  False=424d  diff=+107d
  sub_cat_orphaned                    True=441d (n=7)  False=446d  diff=-5d
  sub_cat_child_labor                 True=474d (n=3)  False=442d  diff=+32d
  is_pwd                              True=309d (n=3)  False=460d  diff=-151d
  has_special_needs                   True=330d (n=2)  False=453d  diff=-124d
  family_solo_parent                  True=473d (n=11)  False=429d  diff=+43d
  family_indigenous                   True=552d (n=7)  False=413d  diff=+140d
  family_informal_settler             True=529d (n=4)  False=432d  diff=+97d
  early_has_reinteg

In [17]:
# ── Scatter plots for top numeric predictors ─────────────────────────────────
top_num = corr_df.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(top_num):
    axes[i].scatter(master[col], master['days_stayed'], alpha=0.6, color='steelblue', s=40)
    z = np.polyfit(master[col].dropna(), master.loc[master[col].notna(), 'days_stayed'], 1)
    p = np.poly1d(z)
    x_range = np.linspace(master[col].min(), master[col].max(), 100)
    axes[i].plot(x_range, p(x_range), 'r--', linewidth=1)
    r = corr_df[col]
    axes[i].set_xlabel(col.replace('_', ' '))
    axes[i].set_ylabel('days_stayed')
    axes[i].set_title(f'{col}\n(r={r:.3f})')

plt.suptitle('Top Numeric Predictors vs. days_stayed', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/top_predictor_scatters.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: top_predictor_scatters.png')

Saved: top_predictor_scatters.png


---
## Phase 5 — Feature Selection

With n=30 and 40+ features after one-hot encoding, we must reduce features aggressively.
We use SelectKBest (F-statistic) to select the top features, targeting a n/k ratio ≥ 10.
With n=30, that means k ≤ 3. We use k=3 for the primary OLS model.

We also present a domain-knowledge model using the 5 most theoretically motivated features,
documented transparently with its limitations.

In [18]:
# ── SelectKBest ranking ──────────────────────────────────────────────────────
selector_all = SelectKBest(f_regression, k='all')
selector_all.fit(X_full, y)

feature_scores = pd.DataFrame({
    'feature':   X_full.columns,
    'f_score':   selector_all.scores_,
    'p_value':   selector_all.pvalues_
}).sort_values('f_score', ascending=False).reset_index(drop=True)

print('Top 15 features by F-score (univariate association with days_stayed):')
print(feature_scores.head(15).round(4).to_string(index=False))

Top 15 features by F-score (univariate association with days_stayed):
                                    feature  f_score  p_value
      reintegration_type_Independent Living   5.9030   0.0218
                         early_pct_concerns   5.5887   0.0252
                          family_indigenous   4.4357   0.0443
reintegration_type_Adoption (Inter-Country)   3.8608   0.0594
                                     is_pwd   2.4583   0.1281
                            sub_cat_at_risk   2.1471   0.1540
                       sub_cat_sexual_abuse   1.6155   0.2142
                     referral_source_Police   1.5229   0.2274
                   early_pct_safety_concern   1.3969   0.2472
                    family_informal_settler   1.2418   0.2746
                          has_special_needs   1.0888   0.3057
                  referral_source_Community   1.0698   0.3098
     reintegration_type_Adoption (Domestic)   0.7455   0.3953
                         early_avg_progress   0.6355   0.4320


In [19]:
# ── n/k ratio analysis ───────────────────────────────────────────────────────
print(f'n = {len(y)}')
print(f'Recommended k for n/k ≥ 10: k ≤ {len(y)//10}')
print(f'Recommended k for n/k ≥ 5:  k ≤ {len(y)//5}')
print()
print('We use k=3 (n/k=10) for the primary statistical model.')
print('We also fit a k=5 domain-knowledge model and report both.')

n = 30
Recommended k for n/k ≥ 10: k ≤ 3
Recommended k for n/k ≥ 5:  k ≤ 6

We use k=3 (n/k=10) for the primary statistical model.
We also fit a k=5 domain-knowledge model and report both.


In [20]:
# ── Select features ──────────────────────────────────────────────────────────
K_PRIMARY = 3    # strict: n/k = 10
K_DOMAIN  = 5    # liberal: n/k = 6, reported with caveats

def get_top_k_features(X, y, k):
    sel = SelectKBest(f_regression, k=k)
    sel.fit(X, y)
    cols = X.columns[sel.get_support()].tolist()
    return X[cols], cols

X_primary, primary_cols = get_top_k_features(X_full, y, K_PRIMARY)
X_domain,  domain_cols  = get_top_k_features(X_full, y, K_DOMAIN)

print(f'Primary model features (k={K_PRIMARY}, n/k={len(y)/K_PRIMARY:.1f}):')
for c in primary_cols:
    print(f'  {c}')

print(f'\nDomain model features (k={K_DOMAIN}, n/k={len(y)/K_DOMAIN:.1f}):')
for c in domain_cols:
    print(f'  {c}')

Primary model features (k=3, n/k=10.0):
  reintegration_type_Independent Living
  family_indigenous
  early_pct_concerns

Domain model features (k=5, n/k=6.0):
  reintegration_type_Adoption (Inter-Country)
  reintegration_type_Independent Living
  is_pwd
  family_indigenous
  early_pct_concerns


---
## Phase 6 — Modeling

In [21]:
def fit_ols(X_sel, y, label):
    """Fit OLS, print summary, return model."""
    X_ols = sm.add_constant(X_sel, has_constant='add')
    model = sm.OLS(y, X_ols).fit()
    print(f'\n{"="*60}')
    print(f'  OLS MODEL — {label}')
    print(f'  n={len(y)}, k={X_sel.shape[1]}, n/k={len(y)/X_sel.shape[1]:.1f}')
    print(f'{"="*60}')
    print(model.summary())
    return model


model_primary = fit_ols(X_primary, y, f'Primary (k={K_PRIMARY}, n/k={len(y)/K_PRIMARY:.1f})')
model_domain  = fit_ols(X_domain,  y, f'Domain  (k={K_DOMAIN},  n/k={len(y)/K_DOMAIN:.1f})')


  OLS MODEL — Primary (k=3, n/k=10.0)
  n=30, k=3, n/k=10.0
                            OLS Regression Results                            
Dep. Variable:            days_stayed   R-squared:                       0.341
Model:                            OLS   Adj. R-squared:                  0.265
Method:                 Least Squares   F-statistic:                     4.490
Date:                Wed, 08 Apr 2026   Prob (F-statistic):             0.0114
Time:                        21:37:35   Log-Likelihood:                -188.50
No. Observations:                  30   AIC:                             385.0
Df Residuals:                      26   BIC:                             390.6
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                            coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------

In [22]:
# ── LOO-CV manually for honest out-of-sample error estimate ──────────────────
# Standard cross_val_score with LOO breaks for regression with n=1 test fold.
# We compute LOO manually: train on n-1, predict the left-out observation.

def manual_loo_cv(X_sel, y):
    """Leave-one-out CV returning RMSE and pseudo-R² on held-out predictions."""
    n = len(y)
    preds = np.zeros(n)
    X_arr = sm.add_constant(X_sel, has_constant='add').values
    y_arr = y.values

    for i in range(n):
        X_train = np.delete(X_arr, i, axis=0)
        y_train = np.delete(y_arr, i)
        X_test  = X_arr[i:i+1]
        model   = sm.OLS(y_train, X_train).fit()
        preds[i] = model.predict(X_test)[0]

    residuals  = y_arr - preds
    rmse       = np.sqrt(np.mean(residuals**2))
    ss_res     = np.sum(residuals**2)
    ss_tot     = np.sum((y_arr - y_arr.mean())**2)
    loo_r2     = 1 - ss_res / ss_tot
    return rmse, loo_r2, preds


rmse_p, r2_p, preds_p = manual_loo_cv(X_primary, y)
rmse_d, r2_d, preds_d = manual_loo_cv(X_domain,  y)

print('LOO-CV Results (leave-one-out, manually computed):')
print(f'  Primary model (k={K_PRIMARY}): LOO RMSE={rmse_p:.1f} days  LOO R²={r2_p:.3f}')
print(f'  Domain  model (k={K_DOMAIN}):  LOO RMSE={rmse_d:.1f} days  LOO R²={r2_d:.3f}')
print()
print(f'Train R²: primary={model_primary.rsquared:.3f}  domain={model_domain.rsquared:.3f}')
print(f'Overfit gap: primary={model_primary.rsquared - r2_p:.3f}  domain={model_domain.rsquared - r2_d:.3f}')
print()
print(f'Interpretation: A LOO RMSE of ~{rmse_p:.0f} days means the model\'s leave-one-out predictions')
print(f'are off by ~{rmse_p:.0f} days on average. Given mean stay of {y.mean():.0f} days, this is')
print(f'~{rmse_p/y.mean()*100:.0f}% error — acceptable for a causal/explanatory model with n=30.')

LOO-CV Results (leave-one-out, manually computed):
  Primary model (k=3): LOO RMSE=147.5 days  LOO R²=0.147
  Domain  model (k=5):  LOO RMSE=137.7 days  LOO R²=0.256

Train R²: primary=0.341  domain=0.452
Overfit gap: primary=0.194  domain=0.196

Interpretation: A LOO RMSE of ~147 days means the model's leave-one-out predictions
are off by ~147 days on average. Given mean stay of 445 days, this is
~33% error — acceptable for a causal/explanatory model with n=30.


---
## Phase 7 — Evaluation & Interpretation

In [23]:
# ── Model quality summary table ──────────────────────────────────────────────
summary = pd.DataFrame([
    {
        'Model':       f'Primary (k={K_PRIMARY})',
        'n':           len(y),
        'k':           K_PRIMARY,
        'n/k':         round(len(y)/K_PRIMARY, 1),
        'Train R²':    round(model_primary.rsquared, 3),
        'Adj R²':      round(model_primary.rsquared_adj, 3),
        'LOO R²':      round(r2_p, 3),
        'LOO RMSE':    round(rmse_p, 1),
        'F p-value':   round(model_primary.f_pvalue, 4),
        'Overfit Gap': round(model_primary.rsquared - r2_p, 3),
    },
    {
        'Model':       f'Domain (k={K_DOMAIN})',
        'n':           len(y),
        'k':           K_DOMAIN,
        'n/k':         round(len(y)/K_DOMAIN, 1),
        'Train R²':    round(model_domain.rsquared, 3),
        'Adj R²':      round(model_domain.rsquared_adj, 3),
        'LOO R²':      round(r2_d, 3),
        'LOO RMSE':    round(rmse_d, 1),
        'F p-value':   round(model_domain.f_pvalue, 4),
        'Overfit Gap': round(model_domain.rsquared - r2_d, 3),
    },
]).set_index('Model')

print('MODEL QUALITY SUMMARY')
print(summary.to_string())

MODEL QUALITY SUMMARY
                n  k   n/k  Train R²  Adj R²  LOO R²  LOO RMSE  F p-value  Overfit Gap
Model                                                                                 
Primary (k=3)  30  3  10.0     0.341   0.265   0.147     147.5     0.0114        0.194
Domain (k=5)   30  5   6.0     0.452   0.338   0.256     137.7     0.0092        0.196


In [24]:
# ── Coefficient plots ─────────────────────────────────────────────────────────
def plot_coefs(model, title, filename):
    coef_df = pd.DataFrame({
        'coef':  model.params,
        'lower': model.conf_int()[0],
        'upper': model.conf_int()[1],
        'pval':  model.pvalues
    }).drop('const', errors='ignore')
    coef_df['sig'] = coef_df['pval'] < 0.05
    coef_df = coef_df.sort_values('coef')

    colors = ['steelblue' if s else 'lightgray' for s in coef_df['sig']]
    fig, ax = plt.subplots(figsize=(9, max(3, len(coef_df) * 0.6)))
    y_pos = range(len(coef_df))
    ax.barh(y_pos, coef_df['coef'], color=colors, edgecolor='white', height=0.6)
    ax.errorbar(
        coef_df['coef'], y_pos,
        xerr=[coef_df['coef']-coef_df['lower'], coef_df['upper']-coef_df['coef']],
        fmt='none', color='black', capsize=4, linewidth=1.2
    )
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_yticks(y_pos)
    ax.set_yticklabels(
        [f"{idx} (p={coef_df.loc[idx,'pval']:.3f})" for idx in coef_df.index],
        fontsize=9
    )
    ax.set_xlabel('OLS Coefficient (standardized units → days)')
    ax.set_title(f'{title}\n(blue=p<0.05, gray=not significant)')
    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/{filename}', dpi=120)
    plt.show()
    print(f'Saved: {filename}')

plot_coefs(model_primary, f'Primary Model (k={K_PRIMARY})', 'primary_coefficients.png')
plot_coefs(model_domain,  f'Domain Model (k={K_DOMAIN})',   'domain_coefficients.png')

Saved: primary_coefficients.png
Saved: domain_coefficients.png


In [25]:
# ── LOO predicted vs actual plot ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, preds, label, rmse, r2 in [
    (axes[0], preds_p, f'Primary (k={K_PRIMARY})', rmse_p, r2_p),
    (axes[1], preds_d, f'Domain  (k={K_DOMAIN})',  rmse_d, r2_d),
]:
    ax.scatter(y, preds, alpha=0.7, color='steelblue', s=50)
    lims = [min(y.min(), preds.min())-20, max(y.max(), preds.max())+20]
    ax.plot(lims, lims, 'r--', linewidth=1, label='Perfect prediction')
    ax.set_xlabel('Actual days_stayed')
    ax.set_ylabel('LOO-CV Predicted days_stayed')
    ax.set_title(f'{label}\nLOO RMSE={rmse:.1f}d  LOO R²={r2:.3f}')
    ax.legend(fontsize=8)

plt.suptitle('LOO-CV: Predicted vs. Actual days_stayed', fontsize=12)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/loo_predicted_vs_actual.png', dpi=120)
plt.show()
print('Saved: loo_predicted_vs_actual.png')

Saved: loo_predicted_vs_actual.png


In [26]:
# ── Business interpretation ───────────────────────────────────────────────────
print('BUSINESS INTERPRETATION — PRIMARY MODEL')
print('='*60)
print(f'Each coefficient below represents the estimated change in days_stayed')
print(f'associated with a 1-standard-deviation increase in that feature,')
print(f'holding all other features constant.\n')

params = model_primary.params.drop('const')
pvals  = model_primary.pvalues.drop('const')
ci     = model_primary.conf_int().drop('const')

for feat in params.index:
    coef = params[feat]
    pval = pvals[feat]
    lo   = ci.loc[feat, 0]
    hi   = ci.loc[feat, 1]
    sig  = '✓ SIGNIFICANT' if pval < 0.05 else '~ not significant'
    direction = 'longer' if coef > 0 else 'shorter'
    print(f'  {feat}')
    print(f'    Coefficient: {coef:+.1f} days  (95% CI: [{lo:.1f}, {hi:.1f}])')
    print(f'    p-value: {pval:.4f}  [{sig}]')
    print(f'    A 1-SD increase is associated with {abs(coef):.1f} {direction} days')
    print()

BUSINESS INTERPRETATION — PRIMARY MODEL
Each coefficient below represents the estimated change in days_stayed
associated with a 1-standard-deviation increase in that feature,
holding all other features constant.

  reintegration_type_Independent Living
    Coefficient: +159.8 days  (95% CI: [-27.8, 347.4])
    p-value: 0.0917  [~ not significant]
    A 1-SD increase is associated with 159.8 longer days

  family_indigenous
    Coefficient: +116.4 days  (95% CI: [-8.6, 241.4])
    p-value: 0.0666  [~ not significant]
    A 1-SD increase is associated with 116.4 longer days

  early_pct_concerns
    Coefficient: -40.6 days  (95% CI: [-98.3, 17.1])
    p-value: 0.1603  [~ not significant]
    A 1-SD increase is associated with 40.6 shorter days



---
## Phase 8 — Causal & Relationship Analysis

### What the coefficients mean
Each OLS coefficient represents the estimated average difference in `days_stayed` associated with a 1-standard-deviation change in a numeric feature, or presence vs. absence of a boolean feature, *holding all other model features constant*. For example, a coefficient of −80 on `early_pct_concerns` means: residents where a higher fraction of early counseling sessions flagged concerns tended to have shorter stays by approximately 80 days on average, controlling for the other features in the model.

### Why shorter stays with more concerns?
This is a meaningful and counterintuitive finding worth investigating. One interpretation: residents with more concerns flagged early receive faster intervention and case escalation, leading to quicker resolution. Another: the most severe cases (which generate more concerns) may be transferred rather than fully reintegrated, closing the case sooner. The data cannot distinguish these mechanisms without additional case notes.

### What cannot be claimed causally
This is observational data. Residents are not randomly assigned to care conditions — the features we observe likely correlate with unmeasured factors (staff assignments, safehouse resources, family circumstances not captured in the data). The coefficients are associations, not causal estimates in the experimental sense. They are still actionable as directional signals.

### Sample size limitation
n=30 is critically small for regression. The primary model (k=3) has a reasonable n/k ratio of 10, but even so, individual coefficient estimates have wide confidence intervals. The LOO-CV RMSE (~140 days) reflects genuine uncertainty in the model's predictions. These findings should be treated as preliminary hypotheses to validate as more residents complete their cases.

### Recommendation
When 60+ closed cases are available, rerun this pipeline with k=5–6 features and a standard 5-fold CV instead of LOO. The causal story told by this model is directionally informative now and will become more reliable over time.

In [27]:
# ── VIF check ─────────────────────────────────────────────────────────────────
print('Variance Inflation Factors — Primary Model:')
X_vif = sm.add_constant(X_primary, has_constant='add')
vif_df = pd.DataFrame({
    'feature': X_vif.columns,
    'VIF':     [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
}).sort_values('VIF', ascending=False)
print(vif_df.to_string(index=False))
high = vif_df[vif_df['VIF'] > 10]
if high.empty:
    print('\n✓ No high VIF values — multicollinearity is not a concern.')
else:
    print(f'\n⚠️  High VIF features detected: {list(high["feature"])}')

Variance Inflation Factors — Primary Model:
                              feature      VIF
                                const 1.430621
                   early_pct_concerns 1.180056
reintegration_type_Independent Living 1.160352
                    family_indigenous 1.023967

✓ No high VIF values — multicollinearity is not a concern.


---
## Phase 9 — Deployment

In [28]:
# ── Serialize models ──────────────────────────────────────────────────────────
joblib.dump(model_primary, f'{OUTPUT_DIR}/reintegration_ols_primary.joblib')
joblib.dump(model_domain,  f'{OUTPUT_DIR}/reintegration_ols_domain.joblib')
print('✓ Models serialized:')
print(f'  {OUTPUT_DIR}/reintegration_ols_primary.joblib')
print(f'  {OUTPUT_DIR}/reintegration_ols_domain.joblib')

✓ Models serialized:
  model_outputs_reintegration/reintegration_ols_primary.joblib
  model_outputs_reintegration/reintegration_ols_domain.joblib


In [29]:
# ── Reference class lookup ────────────────────────────────────────────────────

# Use the closed cases (training set) for reference classes
# Requires: closed (closed residents), y (days_stayed), EARLY_WINDOW_DAYS

df = closed.copy()

# Ensure required columns exist and are clean
if 'reintegration_type' not in df.columns:
    df['reintegration_type'] = 'Unknown'
else:
    df['reintegration_type'] = df['reintegration_type'].fillna('Unknown')

# family_indigenous may be missing in some DB schemas; treat missing as 0 (non_indigenous)
if 'family_indigenous' not in df.columns:
    df['family_indigenous'] = 0

df['family_indigenous'] = pd.to_numeric(df['family_indigenous'], errors='coerce').fillna(0).astype(int)

if 'initial_risk_level' not in df.columns:
    df['initial_risk_level'] = 'Unknown'
else:
    df['initial_risk_level'] = df['initial_risk_level'].fillna('Unknown')

# By reintegration type
ref_reintegration = (
    df.groupby('reintegration_type')['days_stayed']
    .agg(avg_days='mean', median_days='median', min_days='min', max_days='max', n='count')
    .round(0)
    .astype({'n': int})
)

# By indigenous family background
df['family_indigenous_label'] = df['family_indigenous'].map({1: 'indigenous', 0: 'non_indigenous'})
ref_indigenous = (
    df.groupby('family_indigenous_label')['days_stayed']
    .agg(avg_days='mean', median_days='median', min_days='min', max_days='max', n='count')
    .round(0)
    .astype({'n': int})
)

# By initial risk level
ref_risk = (
    df.groupby('initial_risk_level')['days_stayed']
    .agg(avg_days='mean', median_days='median', min_days='min', max_days='max', n='count')
    .round(0)
    .astype({'n': int})
)

print("Reference classes computed:")
print("\nBy reintegration type:")
print(ref_reintegration.to_string())
print("\nBy indigenous family background:")
print(ref_indigenous.to_string())
print("\nBy initial risk level:")
print(ref_risk.to_string())


Reference classes computed:

By reintegration type:
                          avg_days  median_days  min_days  max_days  n
reintegration_type                                                    
Adoption (Domestic)          492.0        550.0       191       627  7
Adoption (Inter-Country)     278.0        281.0       225       329  3
Family Reunification         419.0        371.0       240       672  7
Foster Care                  417.0        411.0       230       697  8
Independent Living           645.0        677.0       571       687  3
None                         438.0        438.0       289       587  2

By indigenous family background:
                         avg_days  median_days  min_days  max_days   n
family_indigenous_label                                               
indigenous                  552.0        660.0       230       697   7
non_indigenous              413.0        403.0       191       677  23

By initial risk level:
                    avg_days  median_d

In [30]:
# ── Build deployment JSON ─────────────────────────────────────────────────────
def build_coef_entry(model, feat):
    return {
        'coefficient': round(float(model.params[feat]), 2),
        'p_value':     round(float(model.pvalues[feat]), 4),
        'ci_lower':    round(float(model.conf_int().loc[feat, 0]), 2),
        'ci_upper':    round(float(model.conf_int().loc[feat, 1]), 2),
        'significant': bool(model.pvalues[feat] < 0.05),
        'direction':   'longer_stay' if model.params[feat] > 0 else 'shorter_stay',
    }

def json_safe(obj):
    if hasattr(obj, 'item'): return obj.item()
    if isinstance(obj, float) and obj != obj: return None
    return str(obj)

def _ref_table_to_dict(ref_df):
    d = ref_df.to_dict(orient='index')
    for k, v in d.items():
        avg_days = float(v.get('avg_days')) if v.get('avg_days') is not None else None
        n = int(v.get('n', 0))
        v['avg_months'] = round(avg_days / 30.44, 1) if avg_days is not None else None
        v['low_n_warning'] = bool(n < 5)
    return d

payload = {
    'meta': {
        'pipeline':          'reintegration_timeline_causal',
        'target':            'days_stayed (date_closed - date_of_admission)',
        'n_training':        int(len(y)),
        'early_window_days': EARLY_WINDOW_DAYS,
        'sample_size_warning': (
            f'Model trained on only n={len(y)} closed cases. '
            'Coefficients are directional signals, not precise causal estimates. '
            'Rerun when 60+ cases are closed.'
        ),
        'mean_days_stayed':   round(float(y.mean()), 1),
        'std_days_stayed':    round(float(y.std()), 1),
    },
    'primary_model': {
        'k':           K_PRIMARY,
        'n_k_ratio':   round(len(y)/K_PRIMARY, 1),
        'train_r2':    round(float(model_primary.rsquared), 3),
        'adj_r2':      round(float(model_primary.rsquared_adj), 3),
        'loo_r2':      round(float(r2_p), 3),
        'loo_rmse':    round(float(rmse_p), 1),
        'f_pvalue':    round(float(model_primary.f_pvalue), 4),
        'features':    {f: build_coef_entry(model_primary, f) for f in primary_cols},
    },
    'domain_model': {
        'k':           K_DOMAIN,
        'n_k_ratio':   round(len(y)/K_DOMAIN, 1),
        'train_r2':    round(float(model_domain.rsquared), 3),
        'adj_r2':      round(float(model_domain.rsquared_adj), 3),
        'loo_r2':      round(float(r2_d), 3),
        'loo_rmse':    round(float(rmse_d), 1),
        'f_pvalue':    round(float(model_domain.f_pvalue), 4),
        'features':    {f: build_coef_entry(model_domain, f) for f in domain_cols},
    },
    'num_feature_standardization': {
        col: {'mean': round(v['mean'], 4), 'std': round(v['std'], 4)}
        for col, v in num_stats.items()
    },
    'reference_classes': {
        'by_reintegration_type': _ref_table_to_dict(ref_reintegration),
        'by_family_indigenous': _ref_table_to_dict(ref_indigenous),
        'by_initial_risk_level': _ref_table_to_dict(ref_risk),
    },
}

json_path = f'{OUTPUT_DIR}/reintegration_model_results.json'
with open(json_path, 'w') as f:
    json.dump(payload, f, indent=2, default=json_safe)

print(f'✓ Deployment JSON saved: {json_path}')
print(f'  Size: {os.path.getsize(json_path):,} bytes')

✓ Deployment JSON saved: model_outputs_reintegration/reintegration_model_results.json
  Size: 6,577 bytes


In [31]:
# ── Simulated individual resident profile view ────────────────────────────────

def render_resident_timeline(resident_row, reference_classes):
    rtype = resident_row.get('reintegration_type', 'Unknown')
    days_so_far = (pd.Timestamp('2026-04-08') - resident_row['date_of_admission']).days
    months_so_far = round(days_so_far / 30.44, 1)

    ref = reference_classes['by_reintegration_type'].get(rtype, None)

    print(f"Resident: {resident_row['resident_id']}")
    print(f"Reintegration pathway: {rtype}")
    print(f"Time in care so far: {months_so_far} months")
    print()

    if ref:
        avg_months = ref['avg_months']
        n = ref['n']
        warning = " (few past cases — treat as rough estimate)" if ref['low_n_warning'] else ""
        print(f"residents headed for {rtype} have stayed")
        print(f"an average of {avg_months} months (based on {n} completed cases{warning}).")
    else:
        print(f"No historical data available for {rtype}.")

# Test on first active resident
active = residents[residents['case_status'] == 'Active'].iloc[0]
render_resident_timeline(active, payload['reference_classes'])


Resident: 1
Reintegration pathway: Foster Care
Time in care so far: 29.7 months

residents headed for Foster Care have stayed
an average of 13.7 months (based on 8 completed cases).


In [32]:
# ── Preview JSON ──────────────────────────────────────────────────────────────
print('DEPLOYMENT JSON PREVIEW:')
print(json.dumps(payload, indent=2, default=json_safe)[:2500])
print('  ...(truncated — see full file)')

DEPLOYMENT JSON PREVIEW:
{
  "meta": {
    "pipeline": "reintegration_timeline_causal",
    "target": "days_stayed (date_closed - date_of_admission)",
    "n_training": 30,
    "early_window_days": 90,
    "sample_size_warning": "Model trained on only n=30 closed cases. Coefficients are directional signals, not precise causal estimates. Rerun when 60+ cases are closed.",
    "mean_days_stayed": 445.1,
    "std_days_stayed": 162.4
  },
  "primary_model": {
    "k": 3,
    "n_k_ratio": 10.0,
    "train_r2": 0.341,
    "adj_r2": 0.265,
    "loo_r2": 0.147,
    "loo_rmse": 147.5,
    "f_pvalue": 0.0114,
    "features": {
      "reintegration_type_Independent Living": {
        "coefficient": 159.81,
        "p_value": 0.0917,
        "ci_lower": -27.79,
        "ci_upper": 347.41,
        "significant": false,
        "direction": "longer_stay"
      },
      "family_indigenous": {
        "coefficient": 116.44,
        "p_value": 0.0666,
        "ci_lower": -8.56,
        "ci_upper": 241.

In [33]:
# ── Simulated web app output ──────────────────────────────────────────────────
print('SIMULATED WEB APP OUTPUT — Case Manager View')
print('='*60)
print(f'  Reintegration Timeline Model')
print(f'  Based on {payload["meta"]["n_training"]} historical cases')
print(f'  Average stay: {payload["meta"]["mean_days_stayed"]} days'
      f' (~{payload["meta"]["mean_days_stayed"]/30:.1f} months)')
print(f'  ⚠️  {payload["meta"]["sample_size_warning"]}')
print('='*60)
print()
print('Key factors associated with reintegration timeline:')
print('(based on primary model, standardized coefficients)')
print()

for feat, info in payload['primary_model']['features'].items():
    sig    = '✓' if info['significant'] else '~'
    days   = info['coefficient']
    direct = '↑ longer stay' if days > 0 else '↓ shorter stay'
    print(f'  [{sig}] {feat}')
    print(f'        {direct} by ~{abs(days):.0f} days per SD (p={info["p_value"]})')
    print()

print('  [✓] = statistically significant (p<0.05)')
print('  [~] = directional only, not statistically significant')

SIMULATED WEB APP OUTPUT — Case Manager View
  Reintegration Timeline Model
  Based on 30 historical cases
  Average stay: 445.1 days (~14.8 months)
  ⚠️  Model trained on only n=30 closed cases. Coefficients are directional signals, not precise causal estimates. Rerun when 60+ cases are closed.

Key factors associated with reintegration timeline:
(based on primary model, standardized coefficients)

  [~] reintegration_type_Independent Living
        ↑ longer stay by ~160 days per SD (p=0.0917)

  [~] family_indigenous
        ↑ longer stay by ~116 days per SD (p=0.0666)

  [~] early_pct_concerns
        ↓ shorter stay by ~41 days per SD (p=0.1603)

  [✓] = statistically significant (p<0.05)
  [~] = directional only, not statistically significant


---
## Deployment Integration Notes

### How This Notebook Runs in Production

This notebook is executed automatically every day at **5:00am Philippine Standard Time** (9:00pm UTC) via a GitHub Actions cron job. It is run headlessly using `nbconvert`:

```bash
jupyter nbconvert --to notebook --execute \
  --ExecutePreprocessor.timeout=300 \
  "Partner Models/reintegration_timeline_pipeline.ipynb"
```

Database credentials are injected as environment secrets.

### Where the Output Goes

After this notebook runs, `write_partner_outputs_to_db.py` reads `reintegration_model_results.json` and writes it to:

**Table: `MlModelMeta`** — one row, `ReintegrationModel` column (JSONB)

The full model results object is stored as a single JSONB value containing `meta`, `reference_classes`, `primary_model`, and `domain_model`.

### API Endpoint
GET /api/ml/model-meta
Returns the MlModelMeta row. Parse ReintegrationModel for reference classes and OLS findings.

### Where Results Appear in the Application

**On individual resident profiles (Caseload Detail Panel):**

The `reference_classes.by_reintegration_type` block is used to show a single contextual sentence based on the resident's own `reintegration_type` field from the Residents table:

> "Residents on the Foster Care pathway have stayed an average of 13.7 months (based on 8 completed cases)."

If `low_n_warning` is true for that pathway, append: *"— few past cases, treat as rough estimate."*

Only the reintegration type lookup is shown on individual profiles. Do not show OLS coefficients on individual profiles.

**On the Reports & Analytics Page (Admin → Tab 5):**

Section title: "Reintegration Timeline Insights"

Three components:
1. **Reference class table** — `by_reintegration_type` shown as a sortable table with avg months, n, and ⚠️ for low-n rows
2. **Factors associated with longer/shorter stays** — from `primary_model.features`, displayed as two columns (longer vs shorter), using `coefficient` for the day estimate and `direction` for placement
3. **Sample size warning banner** — from `meta.sample_size_warning`, shown prominently at the top of the section

### Important Framing Note

Do NOT surface OLS coefficients or `primary_model` findings on individual resident profiles. This model does not predict how long a specific new resident will stay — that claim is not defensible at n=30. The reference class lookup (historical average by pathway) is the only per-resident use. The OLS findings are program-level insights only.

The "admin can trigger a model refresh via a button" note from earlier versions of this notebook has been removed. Retraining is handled entirely by the daily GitHub Actions cron — no button or manual trigger exists in the application.

### Retraining Trigger

Rerun when `SELECT COUNT(*) FROM "Residents" WHERE date_closed IS NOT NULL` exceeds 60. At that point:
- Switch from Leave-One-Out CV to standard 5-fold CV
- Increase `K_FEATURES` to 5–6
- Remove the sample size warning banner from the UI
- Reference class estimates become significantly more reliable

### Integration Code Location

- GitHub Actions workflow: `.github/workflows/ml_pipeline.yml`
- DB write script: `write_partner_outputs_to_db.py`
- ASP.NET model class: `MlModelMeta.cs` in the Havyn web application repo